# SQL Interview Mastery - Practice Notebook

**Kajola Gbenga**
Data Scientist & AI Engineer

*Companion executable notebook to the SQL Interview Mastery handbook. Every pattern here is runnable against a small SQLite practice database built in the first cell, so you can execute, tweak, and re-run each interview pattern instead of just reading it.*

**Dialect note:** SQLite is used as the practice engine because it needs zero setup. A few functions used in the handbook's PostgreSQL-flavored examples (`DATE_TRUNC`, `INTERVAL`, `NOW()`) don't exist in SQLite - this notebook uses SQLite's equivalents (`strftime`, `date(...,'+1 month')`) throughout, and calls out the PostgreSQL/SQL Server/Oracle equivalent in each markdown cell so you can translate on demand (see handbook Part 44).

Run all cells top to bottom.

In [ ]:
import sqlite3, os
import pandas as pd

DB_PATH = "sql_interview_practice.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

def q(sql, params=None):
    """Run a query and return a pandas DataFrame - used throughout this notebook."""
    return pd.read_sql(sql, conn, params=params)

print("SQLite version:", sqlite3.sqlite_version)

## 1. Build the Practice Schema (Handbook Part 30)

Employees/departments, customers, orders/order_items, products, and a generic transactions table - enough to exercise every pattern in the handbook.

In [ ]:
conn.executescript('''
DROP TABLE IF EXISTS order_items; DROP TABLE IF EXISTS orders; DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers; DROP TABLE IF EXISTS employees; DROP TABLE IF EXISTS departments;
DROP TABLE IF EXISTS transactions; DROP TABLE IF EXISTS logins;

CREATE TABLE departments (department_id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY, name TEXT, department_id INTEGER, manager_id INTEGER,
    salary REAL, hire_date TEXT
);
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY, name TEXT, email TEXT UNIQUE, region TEXT, signup_date TEXT
);
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY, name TEXT, category TEXT, unit_price REAL, unit_cost REAL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, status TEXT, total REAL
);
CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY, order_id INTEGER, product_id INTEGER,
    quantity INTEGER, unit_price REAL
);
CREATE TABLE transactions (
    transaction_id INTEGER PRIMARY KEY, account_id INTEGER, amount REAL, txn_type TEXT, txn_date TEXT
);
CREATE TABLE logins (user_id INTEGER, login_date TEXT);
''')
print("Schema created.")

In [ ]:
import random
from datetime import date, timedelta

random.seed(42)

# departments + employees
conn.executescript('''
INSERT INTO departments VALUES (1,'Engineering'), (2,'Sales'), (3,'Marketing');
''')
employees = [
 (1,'Chinedu Okafor',1,None,250000,'2018-03-01'),
 (2,'Amara Nwosu',1,1,180000,'2020-06-15'),
 (3,'Tunde Bakare',1,1,180000,'2021-01-10'),
 (4,'Zainab Bello',2,None,200000,'2017-09-01'),
 (5,'Yusuf Sani',2,4,120000,'2022-04-01'),
 (6,'Ifeoma Chukwu',2,4,135000,'2019-11-20'),
 (7,'Grace Adeyemi',3,None,150000,'2020-02-14'),
 (8,'Emeka Umeh',3,7,95000,'2023-05-05'),
]
conn.executemany("INSERT INTO employees VALUES (?,?,?,?,?,?)", employees)

# customers
regions = ["West","East","North","South"]
customers = [(i, f"Customer {i}", f"customer{i}@example.com", random.choice(regions),
              (date(2024,1,1)+timedelta(days=random.randint(0,600))).isoformat()) for i in range(1,61)]
conn.executemany("INSERT INTO customers VALUES (?,?,?,?,?)", customers)

# products
categories = ["Electronics","Home","Fashion","Sports"]
products = [(i, f"Product {i}", random.choice(categories), round(random.uniform(10,500),2), round(random.uniform(5,300),2))
            for i in range(1,31)]
conn.executemany("INSERT INTO products VALUES (?,?,?,?,?)", products)

# orders + order_items across 2025-2026 with some customers ordering repeatedly (for retention/streak demos)
orders, order_items, oid, oiid = [], [], 1, 1
for cust_id in range(1,61):
    n_orders = random.randint(0,6)
    d = date(2025,1,1) + timedelta(days=random.randint(0,60))
    for _ in range(n_orders):
        d = d + timedelta(days=random.randint(5,45))
        if d > date(2026,8,1): break
        status = random.choices(['completed','cancelled'], weights=[9,1])[0]
        n_items = random.randint(1,4)
        order_total = 0
        item_rows = []
        for _ in range(n_items):
            p = random.choice(products)
            qty = random.randint(1,3)
            item_rows.append((oiid, oid, p[0], qty, p[3]))
            order_total += qty*p[3]
            oiid += 1
        orders.append((oid, cust_id, d.isoformat(), status, round(order_total,2)))
        order_items.extend(item_rows)
        oid += 1

conn.executemany("INSERT INTO orders VALUES (?,?,?,?,?)", orders)
conn.executemany("INSERT INTO order_items VALUES (?,?,?,?,?)", order_items)

# transactions (bank-style, for running-balance / gaps-and-islands demos)
txns, tid = [], 1
for acct in range(1,11):
    d = date(2026,1,1)
    for _ in range(random.randint(10,20)):
        d = d + timedelta(days=random.randint(1,4))
        amt = round(random.uniform(-500,800),2)
        txns.append((tid, acct, amt, 'deposit' if amt>0 else 'withdrawal', d.isoformat()))
        tid += 1
conn.executemany("INSERT INTO transactions VALUES (?,?,?,?,?)", txns)

# logins (for gaps-and-islands streak demo) - a few users with deliberate streaks + gaps
login_rows = []
login_rows += [(101, (date(2026,1,1)+timedelta(days=i)).isoformat()) for i in [0,1,2,5,6,9,10,11,12]]
login_rows += [(102, (date(2026,1,1)+timedelta(days=i)).isoformat()) for i in [0,1,2,3,4,5,6,7]]
conn.executemany("INSERT INTO logins VALUES (?,?)", login_rows)

conn.commit()
print("Employees:", len(employees), "| Customers:", len(customers), "| Products:", len(products))
print("Orders:", len(orders), "| Order items:", len(order_items), "| Transactions:", len(txns))

## 2. The 15-Step Framework in Action (Handbook Part 2)

Worked example: "Find customers whose total 2025-2026 spend is in the top 10% of all customers." Grain: one row per customer. Needs a window function (`NTILE`) since we're bucketing, not just aggregating.

In [ ]:
top_decile = q('''
WITH yearly_spend AS (
    SELECT customer_id, SUM(total) AS total_spend
    FROM orders
    WHERE status = 'completed'
    GROUP BY customer_id
),
ranked AS (
    SELECT customer_id, total_spend,
           NTILE(10) OVER (ORDER BY total_spend DESC) AS decile
    FROM yearly_spend
)
SELECT customer_id, total_spend FROM ranked WHERE decile = 1 ORDER BY total_spend DESC;
''')
top_decile

## 3. Join Traps: LEFT JOIN Neutralized by WHERE (Handbook Part 5, Trap #1)

Compare the row counts: filtering the right table in `WHERE` silently turns a LEFT JOIN into an INNER JOIN.

In [ ]:
bad = q('''
SELECT c.customer_id, o.order_id
FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
''')
good = q('''
SELECT c.customer_id, o.order_id
FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'completed'
''')
print("Rows with filter in WHERE (customers with no completed order silently dropped):", len(bad))
print("Rows with filter in ON (every customer preserved, even with zero completed orders):", len(good))
print("Customers with zero orders at all:", good['order_id'].isna().sum() - bad['order_id'].isna().sum() if 'order_id' in bad else 'n/a')

## 4. Join Fan-Out: Why a LEFT JOIN Can Increase Row Count (Handbook Part 4.4)

Joining `orders` directly to `order_items` multiplies each order row by its item count. Pre-aggregating `order_items` first keeps the join 1:1.

In [ ]:
fan_out = q('''
SELECT o.order_id FROM orders o JOIN order_items oi ON oi.order_id = o.order_id
''')
correct = q('''
SELECT o.order_id, agg.item_count
FROM orders o
JOIN (SELECT order_id, COUNT(*) AS item_count FROM order_items GROUP BY order_id) agg
  ON agg.order_id = o.order_id
''')
print("Row count after naive join (inflated by item count):", len(fan_out))
print("Row count after pre-aggregating order_items first (matches order count):", len(correct))
print("Actual number of orders:", q('SELECT COUNT(*) AS n FROM orders')['n'][0])

## 5. Window Functions Masterclass (Handbook Part 7)

### 5a. Second-highest salary (DENSE_RANK, not ROW_NUMBER - handles ties correctly)

In [ ]:
q('''
SELECT DISTINCT salary FROM (
    SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk FROM employees
) t WHERE rnk = 2
''')

### 5b. Top 2 salaries per department

In [ ]:
q('''
SELECT * FROM (
    SELECT name, department_id, salary,
           DENSE_RANK() OVER (PARTITION BY department_id ORDER BY salary DESC) AS rnk
    FROM employees
) t WHERE rnk <= 2
''')

### 5c. Latest order per customer (ROW_NUMBER - exactly one row even on ties)

In [ ]:
q('''
SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC, order_id DESC) AS rn
    FROM orders
) t WHERE rn = 1
LIMIT 10
''')

### 5d. Running total of daily transaction amounts

In [ ]:
q('''
SELECT txn_date, SUM(amount) AS daily_amount,
       SUM(SUM(amount)) OVER (ORDER BY txn_date ROWS UNBOUNDED PRECEDING) AS running_total
FROM transactions GROUP BY txn_date ORDER BY txn_date
''').head(10)

### 5e. Month-over-month revenue growth
SQLite has no `DATE_TRUNC`; use `strftime('%Y-%m', ...)` instead (PostgreSQL: `DATE_TRUNC('month', order_date)`).

In [ ]:
mom = q('''
WITH m AS (
    SELECT strftime('%Y-%m', order_date) AS month, SUM(total) AS revenue
    FROM orders WHERE status='completed' GROUP BY month
)
SELECT month, revenue,
       revenue - LAG(revenue) OVER (ORDER BY month) AS mom_change,
       ROUND(100.0*(revenue - LAG(revenue) OVER (ORDER BY month)) / NULLIF(LAG(revenue) OVER (ORDER BY month),0), 2) AS mom_pct
FROM m ORDER BY month
''')
mom

In [ ]:
# Quick visual sanity check of the revenue trend used throughout this notebook
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.figure(figsize=(9,4))
plt.plot(mom['month'], mom['revenue'], marker='o')
plt.xticks(rotation=45); plt.title('Monthly Revenue'); plt.tight_layout()
plt.savefig('monthly_revenue.png')
print('Saved monthly_revenue.png')

## 6. Top-N Patterns (Handbook Part 8)

Top 3 customers by total spend, and top 2 products per category by revenue.

In [ ]:
print("Top 3 customers by spend:")
display(q('''
SELECT customer_id, SUM(total) AS spend FROM orders WHERE status='completed'
GROUP BY customer_id ORDER BY spend DESC LIMIT 3
''')) if 'display' in globals() else print(q('''
SELECT customer_id, SUM(total) AS spend FROM orders WHERE status='completed'
GROUP BY customer_id ORDER BY spend DESC LIMIT 3
'''))

In [ ]:
q('''
WITH product_revenue AS (
    SELECT p.product_id, p.category, p.name, SUM(oi.quantity*oi.unit_price) AS revenue
    FROM order_items oi JOIN products p ON p.product_id=oi.product_id
    GROUP BY p.product_id
),
ranked AS (
    SELECT *, RANK() OVER (PARTITION BY category ORDER BY revenue DESC) AS rnk FROM product_revenue
)
SELECT * FROM ranked WHERE rnk <= 2 ORDER BY category, rnk
''')

## 7. Gaps and Islands: Consecutive Login Streaks (Handbook Part 10)

User 101 has two separate streaks with a gap; user 102 has one unbroken 8-day streak. The `date - ROW_NUMBER()` trick isolates each island automatically.

In [ ]:
q('''
WITH numbered AS (
  SELECT user_id, login_date,
         julianday(login_date) - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY login_date) AS grp
  FROM (SELECT DISTINCT user_id, login_date FROM logins) d
)
SELECT user_id, MIN(login_date) AS streak_start, MAX(login_date) AS streak_end, COUNT(*) AS streak_length
FROM numbered GROUP BY user_id, grp ORDER BY user_id, streak_start
''')

## 8. Deduplication: Keep the Latest Record (Handbook Part 11)

Seed a staging table with deliberate duplicate customer emails at different timestamps, then keep only the most recent per email.

In [ ]:
conn.executescript('''
DROP TABLE IF EXISTS customers_staging;
CREATE TABLE customers_staging (staging_id INTEGER PRIMARY KEY, email TEXT, name TEXT, updated_at TEXT);
''')
conn.executemany("INSERT INTO customers_staging (email,name,updated_at) VALUES (?,?,?)", [
    ("dup1@example.com", "Old Name", "2026-01-01"),
    ("dup1@example.com", "New Name", "2026-03-15"),
    ("dup2@example.com", "First",    "2026-02-01"),
    ("dup2@example.com", "Second",   "2026-02-20"),
    ("dup2@example.com", "Third (latest)", "2026-04-01"),
    ("unique@example.com", "Only One", "2026-01-10"),
])
conn.commit()

print("Before dedup:")
print(q("SELECT * FROM customers_staging ORDER BY email, updated_at"))

deduped = q('''
SELECT * FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY email ORDER BY updated_at DESC) AS rn
  FROM customers_staging
) t WHERE rn = 1
''')
print("\nAfter dedup (latest per email survives):")
deduped

## 9. Recursive CTE: Organizational Hierarchy (Handbook Part 13.2)

In [ ]:
q('''
WITH RECURSIVE org_chart AS (
  SELECT employee_id, name, manager_id, 1 AS level
  FROM employees WHERE manager_id IS NULL
  UNION ALL
  SELECT e.employee_id, e.name, e.manager_id, oc.level + 1
  FROM employees e JOIN org_chart oc ON e.manager_id = oc.employee_id
)
SELECT * FROM org_chart ORDER BY level, name
''')

## 10. The NOT IN + NULL Trap (Handbook Part 14, #8)

The single most notorious NULL trap: if the subquery's column contains even one NULL, `NOT IN` returns **zero rows**, always - even when it logically shouldn't.

In [ ]:
# Seed one order with a NULL customer_id to demonstrate the trap
conn.execute("INSERT INTO orders (order_id, customer_id, order_date, status, total) VALUES (99999, NULL, '2026-01-01', 'completed', 10)")
conn.commit()

broken = q('''
SELECT COUNT(*) AS n FROM customers WHERE customer_id NOT IN (SELECT customer_id FROM orders)
''')
fixed = q('''
SELECT COUNT(*) AS n FROM customers c
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
''')
print("NOT IN result (broken by the NULL customer_id row):", broken['n'][0])
print("NOT EXISTS result (correct, NULL-safe):", fixed['n'][0])

# Clean up the seeded NULL row so later cells aren't affected
conn.execute("DELETE FROM orders WHERE order_id = 99999")
conn.commit()

## 11. Conditional Aggregation and Business KPIs (Handbook Parts 17-18)

In [ ]:
kpis = q('''
SELECT
  COUNT(*) AS total_orders,
  SUM(CASE WHEN status='completed' THEN 1 ELSE 0 END) AS completed_orders,
  SUM(CASE WHEN status='cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,
  ROUND(100.0*SUM(CASE WHEN status='cancelled' THEN 1 ELSE 0 END)/COUNT(*), 2) AS cancellation_rate_pct,
  ROUND(SUM(CASE WHEN status='completed' THEN total ELSE 0 END) / NULLIF(SUM(CASE WHEN status='completed' THEN 1 ELSE 0 END),0), 2) AS aov
FROM orders
''')
kpis

In [ ]:
# Gross profit using price-at-time-of-sale (order_items.unit_price), never a "current price" join
gross_profit = q('''
SELECT ROUND(SUM(oi.quantity*oi.unit_price),2) AS revenue,
       ROUND(SUM(oi.quantity*(oi.unit_price - p.unit_cost)),2) AS gross_profit
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.status = 'completed'
''')
gross_profit

## 12. Cohort Retention (Handbook Part 7 #10 / Part 20 #16-17)

Signup-month cohorts, and whether each cohort's customers were also active the following month.

In [ ]:
cohort = q('''
WITH first_order AS (
  SELECT customer_id, MIN(strftime('%Y-%m', order_date)) AS cohort_month FROM orders WHERE status='completed' GROUP BY customer_id
),
monthly_activity AS (
  SELECT DISTINCT customer_id, strftime('%Y-%m', order_date) AS activity_month FROM orders WHERE status='completed'
)
SELECT f.cohort_month, COUNT(DISTINCT f.customer_id) AS cohort_size
FROM first_order f GROUP BY f.cohort_month ORDER BY f.cohort_month
''')
cohort

In [ ]:
retention = q('''
WITH monthly AS (
  SELECT DISTINCT customer_id, strftime('%Y-%m', order_date) AS mo FROM orders WHERE status='completed'
)
SELECT curr.mo,
       COUNT(DISTINCT curr.customer_id) AS active_this_month,
       COUNT(DISTINCT prev.customer_id) AS also_active_prior_month
FROM monthly curr
LEFT JOIN monthly prev
  ON prev.customer_id = curr.customer_id
  AND prev.mo = strftime('%Y-%m', date(curr.mo || '-01', '-1 month'))
GROUP BY curr.mo ORDER BY curr.mo
''')
retention

## 13. Debugging Exercise: Spot and Fix the Bug (Handbook Parts 28-29)

The query below intends to report "average order value per region." Run it, then look closely at the result before checking the fix.

In [ ]:
# BUGGY: joins orders to order_items (fan-out) before averaging - inflates the denominator's row count semantics
buggy = q('''
SELECT c.region, AVG(o.total) AS avg_order_value_buggy, COUNT(*) AS row_count
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
JOIN order_items oi ON oi.order_id = o.order_id   -- <-- the bug: this multiplies rows per order
WHERE o.status='completed'
GROUP BY c.region
''')
buggy

**What's wrong:** joining to `order_items` before aggregating means each order's `total` gets repeated once per line item, so `AVG(o.total)` is still technically averaging the right *values* per repeated row - but `COUNT(*)` no longer reflects the number of orders, and any query relying on that count (or summing `total`) will double/triple count. The fix: aggregate `order_items` separately, or simply don't join to it at all if it isn't needed for this metric.

In [ ]:
fixed = q('''
SELECT c.region, ROUND(AVG(o.total),2) AS avg_order_value, COUNT(*) AS order_count
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
WHERE o.status='completed'
GROUP BY c.region
''')
fixed

## 14. Self-Test: Try It Yourself

Uncomment and complete each query below using the schema built in this notebook. Solutions reference the matching handbook part.

In [ ]:
# 1. Find the second-highest order total per customer region. (Handbook Part 7 #1, adapted)
# your_query = '''...'''
# q(your_query)

# 2. Find customers with a purchase streak of 3+ consecutive months. (Handbook Part 10)
# your_query = '''...'''
# q(your_query)

# 3. Compute month-over-month growth in completed order COUNT (not revenue). (Handbook Part 7 #7 pattern)
# your_query = '''...'''
# q(your_query)

print("Uncomment the lines above and write your own SQL - compare against the matching handbook part.")

## 15. Wrap Up

In [ ]:
conn.execute("PRAGMA optimize;")
conn.close()
print("Connection closed. sql_interview_practice.db retained on disk for further exploration.")
print("See 'SQL Interview Mastery' handbook for the full 65-part reference, question banks, and mock interviews.")